<a href="https://colab.research.google.com/github/Nafeesatu/FitCoach-AI/blob/main/Health_%26_Fitness_Coach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Health & Fitness Coach
Core concept: This is an agent that acts as a personal fitness coach takes user info (age, weight, height, goals, activity level), answers fitness/nutrition questions, generates workout/meal plans, and tracks progress over time.

LLM: Groq because it is free and fast if you don't have paid API credits.

Tools (need ≥2):

Calculator tool — BMI, BMR/TDEE (calorie needs), macro split calculations
Nutrition API — e.g. USDA FoodData Central (free) or Nutritionix, to look up calorie/macro info for foods the user mentions
(optional 3rd) Web search — for things like "latest research on X diet" or exercise form guidance.

Memory:

Short-term: conversation memory within a session (LangChain ConversationBufferMemory or just manual message history)
Long-term: store user profile + logged workouts/meals in a lightweight DB (SQLite or even a JSON file) so the agent "remembers" you across sessions — this is a strong differentiator for your demo

In [107]:
# Install Groq Python SDK for interacting with the Groq API
!pip install -q groq requests sentence-transformers numpy streamlit pyngrok

In [108]:
#  store key using Colab Secrets- good Practice
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get('GROK')
client = Groq(api_key=GROQ_API_KEY)

In [109]:
# This cell tests the basic connection to the Groq API by sending a chat completion request
# and printing the model's response to ensure it's working as expected.
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say hello and confirm you're working as a fitness coach assistant."}
    ]
)
print(response.choices[0].message.content)

Hello. I'm excited to work with you as a fitness coach assistant. I'll be happy to help you with any fitness-related questions, provide workout tips, and support you in achieving your health and wellness goals. How can I assist you today?


In [110]:
%%writefile tools.py
import requests
import sqlite3
from datetime import datetime
import numpy as np

# =========================
# TOOL 1: Calculator
# =========================
def calculate_metrics(weight_kg: float, height_cm: float, age: int, sex: str, activity_level: str, goal: str):
    """Calculate BMI, BMR, TDEE, and recommended daily calories/macros."""
    height_m = height_cm / 100
    bmi = round(weight_kg / (height_m ** 2), 1)

    if sex.lower() == "male":
        bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age + 5
    else:
        bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age - 161

    activity_multipliers = {
        "sedentary": 1.2, "light": 1.375, "moderate": 1.55,
        "active": 1.725, "very_active": 1.9
    }
    multiplier = activity_multipliers.get(activity_level.lower(), 1.2)
    tdee = round(bmr * multiplier)

    if goal == "lose_weight":
        target_calories = tdee - 500
    elif goal == "gain_muscle":
        target_calories = tdee + 300
    else:
        target_calories = tdee

    protein_g = round((target_calories * 0.30) / 4)
    carbs_g = round((target_calories * 0.40) / 4)
    fat_g = round((target_calories * 0.30) / 9)

    return {
        "bmi": bmi, "bmr": round(bmr), "tdee": tdee,
        "target_calories": target_calories,
        "macros": {"protein_g": protein_g, "carbs_g": carbs_g, "fat_g": fat_g}
    }


# =========================
# TOOL 2: Nutrition Lookup
# =========================
NUTRIENT_NUMBERS = {"calories": "208", "protein_g": "203", "fat_g": "204", "carbs_g": "205"}
EXCLUDE_KEYWORDS = ["lunchmeat", "breaded", "roll", "oscar mayer", "deli", "fat-free",
                    "honey", "mesquite", "seasoned", "prepackaged", "rotisserie", "bbq", "tenders"]

def _get_nutrients_by_number(food):
    nutrients = {}
    for n in food["foodNutrients"]:
        num = n.get("nutrientNumber")
        val = n.get("value", 0)
        if num not in nutrients or (nutrients[num] == 0 and val != 0):
            nutrients[num] = val
    if nutrients.get(NUTRIENT_NUMBERS["calories"], 0) <= 0:
        return None
    return nutrients

def lookup_food_nutrition(food_name: str, quantity_grams: float = 100, usda_api_key: str = None):
    """Look up calorie and macro info for a food using USDA FoodData Central (SR Legacy dataset)."""
    search_url = "https://api.nal.usda.gov/fdc/v1/foods/search"
    params = {
        "query": food_name, "api_key": usda_api_key,
        "pageSize": 15, "dataType": ["SR Legacy"]
    }
    resp = requests.get(search_url, params=params)
    data = resp.json()

    if not data.get("foods"):
        return {"error": f"No nutrition data found for '{food_name}'"}

    candidates = [f for f in data["foods"] if not any(kw in f["description"].lower() for kw in EXCLUDE_KEYWORDS)]
    if not candidates:
        candidates = data["foods"]

    def sort_key(f):
        desc = f["description"].lower()
        return (0 if "raw" in desc else 1, len(desc))

    candidates = sorted(candidates, key=sort_key)

    chosen_food, nutrients = None, None
    for f in candidates:
        result = _get_nutrients_by_number(f)
        if result is not None:
            chosen_food, nutrients = f, result
            break

    if chosen_food is None:
        return {"error": f"No usable nutrition data found for '{food_name}'"}

    scale = quantity_grams / 100
    carbs = max(nutrients.get(NUTRIENT_NUMBERS["carbs_g"], 0) * scale, 0)

    return {
        "food": chosen_food["description"],
        "quantity_grams": quantity_grams,
        "calories": round(nutrients.get(NUTRIENT_NUMBERS["calories"], 0) * scale, 1),
        "protein_g": round(nutrients.get(NUTRIENT_NUMBERS["protein_g"], 0) * scale, 1),
        "carbs_g": round(carbs, 1),
        "fat_g": round(nutrients.get(NUTRIENT_NUMBERS["fat_g"], 0) * scale, 1)
    }


# =========================
# TOOLS 3-5: Memory (SQLite)
# =========================
DB_PATH = "fitness_coach.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS user_profile (
            user_id TEXT PRIMARY KEY, weight_kg REAL, height_cm REAL, age INTEGER,
            sex TEXT, activity_level TEXT, goal TEXT, updated_at TEXT
        )
    """)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS progress_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT, user_id TEXT, date TEXT,
            weight_kg REAL, notes TEXT
        )
    """)
    conn.commit()
    conn.close()

def save_user_profile(user_id: str, weight_kg: float, height_cm: float, age: int, sex: str, activity_level: str, goal: str):
    """Save or update a user's profile information."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO user_profile (user_id, weight_kg, height_cm, age, sex, activity_level, goal, updated_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(user_id) DO UPDATE SET
            weight_kg=excluded.weight_kg, height_cm=excluded.height_cm, age=excluded.age,
            sex=excluded.sex, activity_level=excluded.activity_level, goal=excluded.goal,
            updated_at=excluded.updated_at
    """, (user_id, weight_kg, height_cm, age, sex, activity_level, goal, datetime.now().isoformat()))
    conn.commit()
    conn.close()
    return {"status": "profile saved", "user_id": user_id}

def log_progress(user_id: str, weight_kg: float, notes: str = ""):
    """Log a new progress entry for a user."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO progress_log (user_id, date, weight_kg, notes) VALUES (?, ?, ?, ?)
    """, (user_id, datetime.now().strftime("%Y-%m-%d"), weight_kg, notes))
    conn.commit()
    conn.close()
    return {"status": "progress logged", "date": datetime.now().strftime("%Y-%m-%d"), "weight_kg": weight_kg}

def get_user_history(user_id: str):
    """Retrieve a user's saved profile and full progress log history."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM user_profile WHERE user_id = ?", (user_id,))
    profile_row = cursor.fetchone()
    profile = None
    if profile_row:
        profile = {
            "weight_kg": profile_row[1], "height_cm": profile_row[2], "age": profile_row[3],
            "sex": profile_row[4], "activity_level": profile_row[5], "goal": profile_row[6],
            "updated_at": profile_row[7]
        }
    cursor.execute("SELECT date, weight_kg, notes FROM progress_log WHERE user_id = ? ORDER BY date", (user_id,))
    history_rows = cursor.fetchall()
    history = [{"date": r[0], "weight_kg": r[1], "notes": r[2]} for r in history_rows]
    conn.close()

    if not profile and not history:
        return {"message": "No history found for this user yet."}
    return {"profile": profile, "progress_history": history}


# =========================
# TOOL 6: RAG - Fitness Guidelines
# =========================
FITNESS_GUIDELINES = [
    "Beginners should aim for at least 150 minutes of moderate-intensity aerobic activity per week, such as brisk walking, spread across the week rather than done all at once.",
    "Strength training for all major muscle groups should be done at least 2 days per week, allowing at least 48 hours of rest between sessions targeting the same muscle group.",
    "A safe rate of weight loss is generally 0.5 to 1 kg (1 to 2 lbs) per week, achieved through a moderate calorie deficit rather than extreme restriction.",
    "Protein intake for muscle building is generally recommended at 1.6 to 2.2 grams per kilogram of body weight per day, spread across multiple meals.",
    "Proper warm-up before exercise (5-10 minutes of light cardio and dynamic stretching) reduces injury risk and improves performance.",
    "Hydration matters: a general guideline is to drink water before, during, and after exercise, and more in hot conditions or during intense/long sessions.",
    "Rest and recovery are essential -- overtraining without adequate rest can lead to fatigue, decreased performance, and increased injury risk.",
    "Progressive overload -- gradually increasing weight, reps, or intensity over time -- is the key principle behind building strength and muscle.",
    "Sleep of 7-9 hours per night supports muscle recovery, hormone regulation, and overall fitness progress.",
    "Consulting a doctor before starting a new exercise program is recommended for individuals with existing health conditions, injuries, or who have been sedentary for a long time.",
]

_embedder = None
_guideline_embeddings = None

def _get_embedder():
    global _embedder, _guideline_embeddings
    if _embedder is None:
        from sentence_transformers import SentenceTransformer
        _embedder = SentenceTransformer("all-MiniLM-L6-v2")
        _guideline_embeddings = _embedder.encode(FITNESS_GUIDELINES)
    return _embedder, _guideline_embeddings

def retrieve_relevant_guidelines(query: str, top_k: int = 2):
    """Retrieve relevant fitness/nutrition guidelines using semantic similarity search."""
    embedder, guideline_embeddings = _get_embedder()
    query_embedding = embedder.encode([query])

    similarities = np.dot(guideline_embeddings, query_embedding.T).flatten() / (
        np.linalg.norm(guideline_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )
    top_indices = np.argsort(similarities)[::-1][:top_k]
    results = [FITNESS_GUIDELINES[i] for i in top_indices]
    return {"relevant_guidelines": results}

Overwriting tools.py


In [111]:
%%writefile agent.py
import json
from groq import Groq
from tools import (
    calculate_metrics,
    lookup_food_nutrition,
    save_user_profile,
    log_progress,
    get_user_history,
    retrieve_relevant_guidelines
)

MODEL_NAME = "llama-3.3-70b-versatile"

SYSTEM_PROMPT = """You are a knowledgeable, encouraging AI fitness and nutrition coach.
You help users understand their calorie/macro needs, look up nutrition info for foods,
track their progress over time, and give evidence-based fitness guidance.

Guidelines:
- Always be encouraging and supportive, never judgmental about a user's current stats or habits.
- Use tools whenever the user's question requires specific calculations, food data, saved history, or guidelines -- don't guess numbers yourself.
- When a user shares personal stats (weight, height, age, etc.), consider saving their profile using their user_id.
- If a user asks about their progress or past data, retrieve it first using get_user_history before answering.
- Keep responses concise, clear, and actionable.
"""

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate_metrics",
            "description": "Calculate BMI, BMR, TDEE, and recommended daily calories/macros for a user based on their body stats, activity level, and fitness goal.",
            "parameters": {
                "type": "object",
                "properties": {
                    "weight_kg": {"type": "number", "description": "User's weight in kilograms"},
                    "height_cm": {"type": "number", "description": "User's height in centimeters"},
                    "age": {"type": "integer", "description": "User's age in years"},
                    "sex": {"type": "string", "enum": ["male", "female"]},
                    "activity_level": {
                        "type": "string",
                        "enum": ["sedentary", "light", "moderate", "active", "very_active"],
                        "description": "sedentary=little/no exercise, light=1-3 days/week, moderate=3-5 days/week, active=6-7 days/week, very_active=intense daily training"
                    },
                    "goal": {"type": "string", "enum": ["lose_weight", "maintain", "gain_muscle"]}
                },
                "required": ["weight_kg", "height_cm", "age", "sex", "activity_level", "goal"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_food_nutrition",
            "description": "Look up calorie and macronutrient information for a specific food item, scaled to a given quantity in grams.",
            "parameters": {
                "type": "object",
                "properties": {
                    "food_name": {"type": "string", "description": "Name of the food, e.g. 'chicken breast', 'banana'"},
                    "quantity_grams": {"type": "number", "description": "Quantity in grams. Default to 100 if not specified."}
                },
                "required": ["food_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "save_user_profile",
            "description": "Save or update the user's profile information so it can be remembered for future conversations.",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_id": {"type": "string", "description": "Unique identifier for the user"},
                    "weight_kg": {"type": "number"},
                    "height_cm": {"type": "number"},
                    "age": {"type": "integer"},
                    "sex": {"type": "string", "enum": ["male", "female"]},
                    "activity_level": {"type": "string", "enum": ["sedentary", "light", "moderate", "active", "very_active"]},
                    "goal": {"type": "string", "enum": ["lose_weight", "maintain", "gain_muscle"]}
                },
                "required": ["user_id", "weight_kg", "height_cm", "age", "sex", "activity_level", "goal"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "log_progress",
            "description": "Log a new progress entry for the user, such as today's weight and notes.",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_id": {"type": "string"},
                    "weight_kg": {"type": "number"},
                    "notes": {"type": "string", "description": "Optional notes about today's progress"}
                },
                "required": ["user_id", "weight_kg"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_user_history",
            "description": "Retrieve the user's saved profile and full progress log history. Use this whenever the user asks about their progress or past stats.",
            "parameters": {
                "type": "object",
                "properties": {"user_id": {"type": "string"}},
                "required": ["user_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "retrieve_relevant_guidelines",
            "description": "Retrieve relevant evidence-based fitness and nutrition guidelines for general questions (safe weight loss rates, protein needs, recovery, hydration, etc.). Not for calculating a specific user's personal numbers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The user's question or topic"},
                    "top_k": {"type": "integer", "description": "Number of guidelines to retrieve, default 2"}
                },
                "required": ["query"]
            }
        }
    }
]


def make_available_functions(usda_api_key: str):
    """Bind the USDA API key into lookup_food_nutrition via a wrapper, and return the full registry."""
    def _lookup_wrapper(food_name, quantity_grams=100):
        return lookup_food_nutrition(food_name, quantity_grams, usda_api_key=usda_api_key)

    return {
        "calculate_metrics": calculate_metrics,
        "lookup_food_nutrition": _lookup_wrapper,
        "save_user_profile": save_user_profile,
        "log_progress": log_progress,
        "get_user_history": get_user_history,
        "retrieve_relevant_guidelines": retrieve_relevant_guidelines
    }


def run_agent(client: Groq, messages: list, available_functions: dict):
    """
    Runs the tool-calling agent loop:
    sends messages -> checks for tool calls -> executes them -> feeds results back
    -> repeats until the model returns a final text answer.
    """
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto"
    )
    response_message = response.choices[0].message
    messages.append(response_message)

    tool_call_log = []

    while response_message.tool_calls:
        for tool_call in response_message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)

            tool_call_log.append({"tool": func_name, "args": func_args})

            function_to_call = available_functions[func_name]
            function_result = function_to_call(**func_args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(function_result)
            })

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto"
        )
        response_message = response.choices[0].message
        messages.append(response_message)

    return response_message.content, messages, tool_call_log

Overwriting agent.py


In [112]:
%%writefile app.py
import streamlit as st
from groq import Groq
from agent import run_agent, make_available_functions, SYSTEM_PROMPT
from tools import init_db

st.set_page_config(
    page_title="FitCoach AI",
    page_icon="💪",
    layout="centered",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
    .stApp {
        background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
    }
    .main-header {
        text-align: center;
        padding: 1.5rem 0 0.5rem 0;
    }
    .main-header h1 {
        color: #ffffff;
        font-size: 2.2rem;
        font-weight: 800;
        margin-bottom: 0.2rem;
    }
    .main-header p {
        color: #94a3b8;
        font-size: 1rem;
        margin-top: 0;
    }
    .stChatMessage {
        background-color: rgba(255, 255, 255, 0.04);
        border-radius: 16px;
        padding: 0.5rem 0.2rem;
        margin-bottom: 0.5rem;
        border: 1px solid rgba(255, 255, 255, 0.06);
    }
    section[data-testid="stSidebar"] {
        background-color: #0b1220;
        border-right: 1px solid rgba(255,255,255,0.08);
    }
    section[data-testid="stSidebar"] h1,
    section[data-testid="stSidebar"] h2,
    section[data-testid="stSidebar"] h3 {
        color: #f1f5f9;
    }
    section[data-testid="stSidebar"] p,
    section[data-testid="stSidebar"] li {
        color: #cbd5e1;
    }
    .stTextInput input {
        border-radius: 10px;
        border: 1px solid rgba(255,255,255,0.15);
        background-color: rgba(255,255,255,0.05);
        color: white;
    }
    .stChatInput textarea {
        border-radius: 12px !important;
    }
    .feature-pill {
        display: inline-block;
        background: rgba(99, 102, 241, 0.15);
        color: #a5b4fc;
        padding: 4px 12px;
        border-radius: 20px;
        font-size: 0.8rem;
        margin: 3px 3px 3px 0;
        border: 1px solid rgba(99, 102, 241, 0.3);
    }
    .streamlit-expanderHeader {
        background-color: rgba(99, 102, 241, 0.08) !important;
        border-radius: 10px !important;
        color: #a5b4fc !important;
    }
    #MainMenu {visibility: hidden;}
    footer {visibility: hidden;}
</style>
""", unsafe_allow_html=True)

init_db()

with st.sidebar:
    st.markdown("## 💪 FitCoach AI")
    st.markdown("Your personal AI-powered fitness and nutrition coach.")
    st.markdown("---")

    user_id = st.text_input(
        "👤 Your name or ID",
        value=st.session_state.get("user_id", ""),
        placeholder="e.g. Nafisat",
        help="Used to remember your profile and progress across sessions."
    )
    st.session_state["user_id"] = user_id

    st.markdown("---")
    st.markdown("**What I can help with:**")
    st.markdown("""
        <span class="feature-pill">📊 Calorie & macro calculator</span>
        <span class="feature-pill">🍗 Food nutrition lookup</span>
        <span class="feature-pill">📈 Progress tracking</span>
        <span class="feature-pill">🧠 Evidence-based guidance</span>
    """, unsafe_allow_html=True)

    st.markdown("---")
    if st.session_state.get("user_id"):
        if st.button("🗑️ Clear conversation", use_container_width=True):
            st.session_state.messages = [{"role": "system", "content": SYSTEM_PROMPT}]
            st.rerun()

    st.markdown("---")
    st.caption("Built with Groq (Llama 3.3), Streamlit, USDA FoodData Central, and RAG.")

try:
    GROQ_API_KEY = st.secrets["GROQ_API_KEY"]
    USDA_API_KEY = st.secrets["USDA_API_KEY"]
except Exception:
    st.error("API keys not found. Please configure GROQ_API_KEY and USDA_API_KEY in Streamlit secrets.")
    st.stop()

client = Groq(api_key=GROQ_API_KEY)
available_functions = make_available_functions(usda_api_key=USDA_API_KEY)

st.markdown("""
<div class="main-header">
    <h1>💪 FitCoach AI</h1>
    <p>Personalized fitness coaching, powered by AI</p>
</div>
""", unsafe_allow_html=True)

if not user_id:
    st.info("👈 Please enter your name or ID in the sidebar to get started.")
    st.stop()

if "messages" not in st.session_state:
    st.session_state.messages = [{"role": "system", "content": SYSTEM_PROMPT}]

has_visible_messages = any(
    (msg["role"] if isinstance(msg, dict) else msg.role) == "user"
    for msg in st.session_state.messages
)

if not has_visible_messages:
    with st.chat_message("assistant", avatar="💪"):
        st.write(
            f"Hey {user_id}! 👋 I'm your AI fitness coach. Tell me a bit about "
            "yourself (age, weight, height, activity level, and goal) and I can "
            "calculate your calorie and macro targets, look up nutrition info for "
            "any food, and track your progress over time. What would you like to start with?"
        )

for msg in st.session_state.messages:
    role = msg["role"] if isinstance(msg, dict) else msg.role
    content = msg["content"] if isinstance(msg, dict) else msg.content

    if role == "user":
        with st.chat_message("user", avatar="🧑"):
            display_content = content
            if display_content.startswith("[user_id:"):
                display_content = display_content.split("]", 1)[-1].strip()
            st.write(display_content)
    elif role == "assistant" and content:
        with st.chat_message("assistant", avatar="💪"):
            st.write(content)

user_input = st.chat_input("Ask me about calories, nutrition, or your progress...")

if user_input:
    contextual_input = f"[user_id: {user_id}] {user_input}"
    st.session_state.messages.append({"role": "user", "content": contextual_input})

    with st.chat_message("user", avatar="🧑"):
        st.write(user_input)

    with st.chat_message("assistant", avatar="💪"):
        with st.spinner("Thinking..."):
            answer, updated_messages, tool_log = run_agent(
                client, st.session_state.messages, available_functions
            )
            st.session_state.messages = updated_messages
            st.write(answer)

            if tool_log:
                with st.expander("🔧 Tools used in this response"):
                    for call in tool_log:
                        st.write(f"**{call['tool']}**({call['args']})")

Overwriting app.py


In [113]:
%%writefile requirements.txt
streamlit
groq
requests
sentence-transformers
numpy

Overwriting requirements.txt


In [114]:
# validate all files
import ast

for filename in ["tools.py", "agent.py", "app.py"]:
    with open(filename) as f:
        source = f.read()
    try:
        ast.parse(source)
        print(f"✅ {filename} is valid ({len(source.splitlines())} lines)")
    except SyntaxError as e:
        print(f"❌ {filename} has a SyntaxError: {e}")

✅ tools.py is valid (218 lines)
✅ agent.py is valid (188 lines)
✅ app.py is valid (187 lines)


In [115]:
# Create local secrets file for testing in Colab
import os
os.makedirs(".streamlit", exist_ok=True)

with open(".streamlit/secrets.toml", "w") as f:
    f.write(f'GROQ_API_KEY = "{GROQ_API_KEY}"\n')
    f.write(f'USDA_API_KEY = "{USDA_API_KEY}"\n')

print("secrets.toml created.")

secrets.toml created.


In [116]:
# Authenticate ngrok
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

##End Part 1

In [117]:
import subprocess, time
from pyngrok import ngrok

# Kill any previous streamlit process
!pkill -f streamlit

# Kill any existing ngrok tunnels
ngrok.kill()

# Start Streamlit in the background
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=open("streamlit_log.txt", "a"),
    stderr=subprocess.STDOUT
)

time.sleep(5)  # give it a moment to start

# Open the tunnel
public_url = ngrok.connect(8501)
print("Your app is live at:", public_url)

Your app is live at: NgrokTunnel: "https://landmark-cucumber-disaster.ngrok-free.dev" -> "http://localhost:8501"
